In [2]:
import os
import csv
import gzip
from glob import glob

# ============================================================
# CONFIG
# ============================================================
SEARCH_DIRS = [
    "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/EnRich/20250124/Delly_012425",
    "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/EnRich/20260105/Delly_010526",
    "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/EnRich/20260213/Batch2_021326/Delly_021326",
]

FUSION_LIST = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/EnRich/Delly/Survivor_fusion_list.txt"
OUTDIR = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/EnRich/Delly"

FULL_MATCH_OUT = os.path.join(OUTDIR, "delly_fusion_full_matches.tsv")
MISSING_OUT    = os.path.join(OUTDIR, "delly_fusion_missing_samples_or_no_match.tsv")

# fusion list column names
SAMPLE_COL = "Sample"
CHR1_COL   = "ChrA"
CHR2_COL   = "ChrB"

# ============================================================
# HELPERS
# ============================================================
def norm_chr(chrom):
    chrom = str(chrom).strip()
    if not chrom.startswith("chr"):
        chrom = "chr" + chrom
    return chrom

def parse_info(info_str):
    d = {}
    for item in info_str.split(";"):
        if "=" in item:
            k, v = item.split("=", 1)
            d[k] = v
        else:
            d[item] = True
    return d

def open_any_vcf(path):
    if path.endswith(".gz"):
        return gzip.open(path, "rt")
    return open(path, "r")

def index_vcfs(search_dirs):
    sample_to_vcf = {}
    duplicates = {}

    for base in search_dirs:
        patterns = [
            os.path.join(base, "*", "*_delly_filtered_merged.vcf"),
            os.path.join(base, "*", "*_delly_filtered_merged.vcf.gz"),
        ]
        for pattern in patterns:
            for vcf in glob(pattern):
                sample = os.path.basename(os.path.dirname(vcf))
                if sample in sample_to_vcf:
                    duplicates.setdefault(sample, []).append(vcf)
                else:
                    sample_to_vcf[sample] = vcf

    return sample_to_vcf, duplicates

def find_matches_in_vcf(vcf_path, sample, query_chr1, query_chr2):
    matches = []

    with open_any_vcf(vcf_path) as fh:
        for line in fh:
            if line.startswith("##"):
                continue
            if line.startswith("#CHROM"):
                continue

            parts = line.rstrip("\n").split("\t")
            if len(parts) < 10:
                continue

            chrom, pos, vid, ref, alt, qual, filt, info, fmt, sample_field = parts[:10]
            info_dict = parse_info(info)

            if info_dict.get("SVTYPE", "") != "BND":
                continue

            chr2 = info_dict.get("CHR2", "")
            pos2 = info_dict.get("POS2", "")
            ct   = info_dict.get("CT", "")
            pe   = info_dict.get("PE", "")
            sr   = info_dict.get("SR", "")

            if chr2 == "":
                continue

            chrom = norm_chr(chrom)
            chr2  = norm_chr(chr2)

            # unordered chromosome-pair match
            if {chrom, chr2} == {query_chr1, query_chr2}:
                matches.append({
                    "query_sample": sample,
                    "query_chr1": query_chr1,
                    "query_chr2": query_chr2,
                    "source_vcf": vcf_path,
                    "#CHROM": chrom,
                    "POS": pos,
                    "ID": vid,
                    "REF": ref,
                    "ALT": alt,
                    "QUAL": qual,
                    "FILTER": filt,
                    "INFO": info,
                    "FORMAT": fmt,
                    "SAMPLE_FIELD": sample_field,
                    "CHR2": chr2,
                    "POS2": pos2,
                    "CT": ct,
                    "PE": pe,
                    "SR": sr
                })
    return matches

# ============================================================
# MAIN
# ============================================================
def main():
    os.makedirs(OUTDIR, exist_ok=True)

    sample_to_vcf, duplicates = index_vcfs(SEARCH_DIRS)

    if duplicates:
        print("[WARNING] Duplicate sample folders found. Using first occurrence:")
        for s, paths in duplicates.items():
            print(f"  {s}")
            print(f"    kept : {sample_to_vcf[s]}")
            for p in paths:
                print(f"    extra: {p}")

    full_rows = []
    missing_rows = []

    with open(FUSION_LIST, "r", newline="") as f:
        reader = csv.DictReader(f, delimiter="\t")

        required = {SAMPLE_COL, CHR1_COL, CHR2_COL}
        if not required.issubset(reader.fieldnames):
            raise ValueError(
                f"Fusion list must contain columns: {required}. Found: {reader.fieldnames}"
            )

        for row in reader:
            sample = row[SAMPLE_COL].strip()
            chr1   = norm_chr(row[CHR1_COL])
            chr2   = norm_chr(row[CHR2_COL])

            if sample not in sample_to_vcf:
                missing_rows.append({
                    "sample": sample,
                    "chr1": chr1,
                    "chr2": chr2,
                    "status": "sample_vcf_not_found"
                })
                continue

            vcf_path = sample_to_vcf[sample]
            matches = find_matches_in_vcf(vcf_path, sample, chr1, chr2)

            if not matches:
                missing_rows.append({
                    "sample": sample,
                    "chr1": chr1,
                    "chr2": chr2,
                    "status": "no_matching_bnd_found"
                })
                continue

            full_rows.extend(matches)

    # write full matched rows
    with open(FULL_MATCH_OUT, "w", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "query_sample", "query_chr1", "query_chr2", "source_vcf",
                "#CHROM", "POS", "ID", "REF", "ALT", "QUAL", "FILTER",
                "INFO", "FORMAT", "SAMPLE_FIELD", "CHR2", "POS2", "CT", "PE", "SR"
            ],
            delimiter="\t"
        )
        writer.writeheader()
        writer.writerows(full_rows)

    # write missing file only if there are real missing/unmatched rows
    if missing_rows:
        with open(MISSING_OUT, "w", newline="") as f:
            writer = csv.DictWriter(
                f,
                fieldnames=["sample", "chr1", "chr2", "status"],
                delimiter="\t"
            )
            writer.writeheader()
            writer.writerows(missing_rows)
        print(f"Missing/unmatched file written: {MISSING_OUT}")
    else:
        if os.path.exists(MISSING_OUT):
            os.remove(MISSING_OUT)
        print("[INFO] No missing/unmatched entries. Missing file not created.")

    print("[DONE]")
    print(f"Full matches : {FULL_MATCH_OUT}")
    print(f"Matched rows : {len(full_rows)}")
    print(f"Missing rows : {len(missing_rows)}")

if __name__ == "__main__":
    main()

[WARNING] Duplicate sample folders found. Using first occurrence:
  TP20_M63
    kept : /ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/EnRich/20250124/Delly_012425/TP20_M63/TP20_M63_delly_filtered_merged.vcf
    extra: /ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/EnRich/20250124/Delly_012425/TP20_M63/TP20_M63_delly_filtered_merged.vcf.gz
  TP22_M34
    kept : /ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/EnRich/20250124/Delly_012425/TP22_M34/TP22_M34_delly_filtered_merged.vcf
    extra: /ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/EnRich/20250124/Delly_012425/TP22_M34/TP22_M34_delly_filtered_merged.vcf.gz
  TP23_M531
    kept : /ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/EnRich/20250124/Delly_012425/TP23_M531/TP23_M531_delly_filtered_merged.vcf
    extra: /ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/EnRich/20250124/Delly_012425/TP23_M531/TP23_M531_delly_filtered_merged.vcf.gz
  TP22_M617
    kept : /ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/EnRich/20250124/Delly_012425/TP22_M617